# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder) without this

/Users/mac/Documents/dev/ID2221/dic/Week 2


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
from Queries import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bf88883f-4c21-4259-b107-72e64a234dad;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delt

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Ensure no auto broadcast

In [3]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# reset the broadcast threshold to default value
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10 MB

# Disable AQE

In [4]:
# disable AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")

# Uncache Tables

In [5]:
spark.catalog.uncacheTable("default.air_quality")
spark.catalog.uncacheTable("default.taxi_trips")
spark.catalog.uncacheTable("default.taxi_zone_lookup")
spark.catalog.uncacheTable("default.weather")

## Run queries on uncached tables

### Query 2.1

In [37]:
result = spark.sql(query_2_1())
result.summary().show()
result.show()
result.explain(mode="formatted")


+-------+--------------------+------------------+------------------+
|summary|             pu_zone|             month|         row_count|
+-------+--------------------+------------------+------------------+
|  count|                 270|               271|               271|
|   mean|                NULL|1.4575645756457565|10939.365313653136|
| stddev|                NULL| 2.174994246582365|27353.533786659147|
|    min|Allerton/Pelham G...|                 1|                 1|
|    25%|                NULL|                 1|                78|
|    50%|                NULL|                 1|               245|
|    75%|                NULL|                 1|              1393|
|    max|      Yorkville West|                12|            145240|
+-------+--------------------+------------------+------------------+

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|            Ki

### Query 2.2

In [38]:
result = spark.sql(query_2_2())
result.show()
result.explain(mode="formatted")


+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Physical Plan ==
* HashAggregate (14)
+- Exchange (13)
   +- * HashAggregate (12)
      +- * Project (11)
         +- * SortMergeJoin LeftOuter (10)
            :- * Sort (4)
            :  +- Exchange (3)
            :     +- * ColumnarToRow (2)
            :        +- Scan parquet spark_catalog.default.taxi_trips (1)
            +- * Sort (9)
               +- Exchange (8)
                  +- * Filter (7)
                     +- * ColumnarToRow (6)
                        +- Scan parquet spark_catalog.default.weather (5)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [2]: [pu_datetime#34330, trip_distance#34335]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_pr

### Query 2.3

In [39]:
res = spark.sql(query_2_3())
res.show()
res.explain(mode="formatted")

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Physical Plan ==
* Sort (33)
+- Exchange (32)
   +- * HashAggregate (31)
      +- Exchange (30)
         +- * HashAggregate (29)
            +- * Project (28)
               +- * SortMergeJoin LeftOuter (27)
                  :- * Sort (13)
                  :  +- Exchange (12)
                  :     +- * Project (11)
                  :        +- * SortMergeJoin LeftOuter (10)
                  :           :- * Sort (4)
                  

### Query 2.4

In [44]:
res = spark.sql(query_2_4())
res.show()
res.explain(mode="formatted")

+---------+------------+-------------+--------------+
|   county|weather_cond|weather_hours|trips_per_hour|
+---------+------------+-------------+--------------+
|    Bronx|         dry|            2|           6.0|
|    Bronx|        rain|           10|           8.9|
|    Bronx|       humid|          210|          9.97|
|    Bronx|        cold|          173|         10.08|
|    Bronx|      stormy|          259|         10.45|
|    Bronx|    moderate|           24|         10.83|
| Brooklyn|        rain|           11|         23.45|
| Brooklyn|         dry|            2|          23.5|
| Brooklyn|    moderate|           26|         32.65|
| Brooklyn|       humid|          224|         33.55|
| Brooklyn|      stormy|          294|         34.36|
| Brooklyn|        cold|          186|         34.85|
|Manhattan|        rain|           11|        1719.0|
|Manhattan|    moderate|           26|       2757.77|
|Manhattan|         dry|            2|        3464.0|
|Manhattan|       humid|    

### Query 2.5

In [40]:
res = spark.sql(query_2_5())
res.show()
res.explain(mode="formatted")

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows
== Physical Plan ==
* Sort (8)
+- Exchange (7)
   +- * HashAggregate (6)
      +- Exchange (5)
         +- * HashAggregate (4)
            +- * Project (3)
               +- * ColumnarToRow (2)
                  +- Scan parquet spark_catalog.default.taxi_trips (1)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [1]: [pu_datetime#35157]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: struct<pu_datetime:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [1]

### Query 2.6

In [41]:
res = spark.sql(query_2_6())
res.show()
res.explain(mode="formatted")

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+

== Physical Plan ==
* Sort (8)
+- Exchange (7)
   +- * HashAggregate (6)
      +- Exchange (5)
         +- * HashAggregate (4)
            +- * Project (3)
               +- * ColumnarToRow (2)
                  +- Scan parquet spark_catalog.default.taxi_trips (1)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [1]: [pu_datetime#35339]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: struct<pu_datetime:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [pu_datetime#35339]

(3) Project [codegen id : 1]
Output [1]: [date_format(pu_datetime#35339, MMM, Some(Europe/Stockholm)) AS _groupingexpression#35435]
Input [1]: [pu_datetime#35339]

(4) HashAggregate [codegen id : 1]
Input [1]: [_groupingexpression#35435]
Keys [1]: [_groupingexpression#35435]
Functions [1]: [parti

# Cache Tables

In [45]:
spark.catalog.cacheTable("default.air_quality")
spark.catalog.cacheTable("default.taxi_trips")
spark.catalog.cacheTable("default.taxi_zone_lookup")
spark.catalog.cacheTable("default.weather")

## Run queries on cached tables

### Query 2.1

In [66]:
result = spark.sql(query_2_1())
result.summary().show()
result.show()
result.explain(mode="formatted")


+-------+--------------------+------------------+------------------+
|summary|             pu_zone|             month|         row_count|
+-------+--------------------+------------------+------------------+
|  count|                 270|               271|               271|
|   mean|                NULL|1.4575645756457565|10939.365313653136|
| stddev|                NULL| 2.174994246582365|27353.533786659147|
|    min|Allerton/Pelham G...|                 1|                 1|
|    25%|                NULL|                 1|                78|
|    50%|                NULL|                 1|               245|
|    75%|                NULL|                 1|              1393|
|    max|      Yorkville West|                12|            145240|
+-------+--------------------+------------------+------------------+

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|            Ki

### Query 2.2

In [67]:
result = spark.sql(query_2_2())
result.show()
result.explain(mode="formatted")


+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Physical Plan ==
* HashAggregate (18)
+- Exchange (17)
   +- * HashAggregate (16)
      +- * Project (15)
         +- * SortMergeJoin LeftOuter (14)
            :- * Sort (6)
            :  +- Exchange (5)
            :     +- Scan In-memory table default.taxi_trips (1)
            :           +- InMemoryRelation (2)
            :                 +- * ColumnarToRow (4)
            :                    +- Scan parquet spark_catalog.default.taxi_trips (3)
            +- * Sort (13)
               +- Exchange (12)
                  +- * Filter (11)
                     +- Scan In-memory table default.weather (7)
                           +- InMemoryRelation (8)
                                 +- * ColumnarToRow (10)
       

### Query 2.3

In [62]:
res = spark.sql(query_2_3())
res.show()
res.explain(mode="formatted")

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Physical Plan ==
* Sort (39)
+- Exchange (38)
   +- * HashAggregate (37)
      +- Exchange (36)
         +- * HashAggregate (35)
            +- * Project (34)
               +- * SortMergeJoin LeftOuter (33)
                  :- * Sort (17)
                  :  +- Exchange (16)
                  :     +- * Project (15)
                  :        +- * SortMergeJoin LeftOuter (14)
                  :           :- * Sort (6)
                  

### Query 2.4

In [63]:
res = spark.sql(query_2_4())
res.show()
res.explain(mode="formatted")

+---------+------------+-------------+--------------+
|   county|weather_cond|weather_hours|trips_per_hour|
+---------+------------+-------------+--------------+
|    Bronx|         dry|            2|           6.0|
|    Bronx|        rain|           10|           8.9|
|    Bronx|       humid|          210|          9.97|
|    Bronx|        cold|          173|         10.08|
|    Bronx|      stormy|          259|         10.45|
|    Bronx|    moderate|           24|         10.83|
| Brooklyn|        rain|           11|         23.45|
| Brooklyn|         dry|            2|          23.5|
| Brooklyn|    moderate|           26|         32.65|
| Brooklyn|       humid|          224|         33.55|
| Brooklyn|      stormy|          294|         34.36|
| Brooklyn|        cold|          186|         34.85|
|Manhattan|        rain|           11|        1719.0|
|Manhattan|    moderate|           26|       2757.77|
|Manhattan|         dry|            2|        3464.0|
|Manhattan|       humid|    

### Query 2.5

In [64]:
res = spark.sql(query_2_5())
res.show()
res.explain(mode="formatted")

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows
== Physical Plan ==
* Sort (10)
+- Exchange (9)
   +- * HashAggregate (8)
      +- Exchange (7)
         +- * HashAggregate (6)
            +- * Project (5)
               +- Scan In-memory table default.taxi_trips (1)
                     +- InMemoryRelation (2)
                           +- * ColumnarToRow (4)
                              +- Scan parquet spark_catalog.default.taxi_trips (3)


(1) Scan In-memory table default.taxi_trips
Output [1]: [pu_datetime#63347]
Arguments: [pu_datetime#63347]

(2) InMemoryRelation
Arguments: [pu_datetime#63347, do_datet

### Query 2.6

In [65]:
res = spark.sql(query_2_6())
res.show()
res.explain(mode="formatted")

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+

== Physical Plan ==
* Sort (10)
+- Exchange (9)
   +- * HashAggregate (8)
      +- Exchange (7)
         +- * HashAggregate (6)
            +- * Project (5)
               +- Scan In-memory table default.taxi_trips (1)
                     +- InMemoryRelation (2)
                           +- * ColumnarToRow (4)
                              +- Scan parquet spark_catalog.default.taxi_trips (3)


(1) Scan In-memory table default.taxi_trips
Output [1]: [pu_datetime#63643]
Arguments: [pu_datetime#63643]

(2) InMemoryRelation
Arguments: [pu_datetime#63643, do_datetime#63644, pu_location_id#63645, do_location_id#63646, fare_amount#63647, trip_distance#63648], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan parquet spark_catalog.default.taxi_trips
Output [6]: [pu_datetime#37881, do_datetime#37882, pu_location_id#37883, do_location_id#37884, fare_amount#37885, trip_d